# Hidden Regimes
### Autoencoder + Gaussian HMM market regime detection

**Objective.** Detect and characterize market regimes from a diverse macro/vol/credit feature set. This project has **no P&L objective** — unlike the two sibling projects in this series (a cross-sectional equity reversal strategy and an ETF-basket arbitrage strategy, both of which were rigorously backtested and both of which turned out to be statistically insignificant after correcting for multiple testing), this one is a standalone research model. It's validated on its own statistical and economic merits: does it generalize out of sample, does it find regimes that line up with reality, and is it stable when refit on more data.

**Pipeline:** raw macro/vol/credit features → a small autoencoder compresses them into a learned 3-dimensional embedding → a Gaussian HMM fits regimes on top of that embedding. Representation learning feeding a generative sequence model, rather than either alone — and a genuinely different architecture from the gradient-boosted-tree classifiers used in the other two projects.

An interactive dashboard built from this same model is published separately; this notebook is the narrative walkthrough of how it was built and validated.

In [ ]:
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x: .4f}")

import config
from data.prices import load_many
from features.engineering import build_raw_features
from models.autoencoder import train_autoencoder
from models.regime_hmm import sweep_n_states, select_n_states, implied_durations, predict_regimes

print("modules loaded")

## 1 · Data and features

**Universe:** a deliberately diverse set of 12 tickers, not just equity prices — the whole point of using an autoencoder later is to find structure across genuinely different inputs, so the more different they are, the more interesting the learned embedding:

| Ticker | Role |
|---|---|
| `SPY` | broad equity |
| `RSP` | equal-weight equity — vs `SPY` gives a breadth/concentration proxy |
| `IWM` | small caps — risk-appetite proxy |
| `QQQ` | growth/tech — style-rotation proxy |
| `^VIX` | implied vol |
| `HYG` / `LQD` | high-yield vs. investment-grade credit — the spread is a credit-stress proxy |
| `TLT` | long treasuries — rates/duration proxy |
| `GLD` | gold — safe-haven flows |
| `UUP` | dollar index — USD flows |
| `USO` | oil — commodity vol proxy |
| `^TNX` | 10-year yield level |

**Start date: 2007-01-01**, chosen specifically so the 2008 financial crisis — the single most useful "does this model actually work" sanity check available — falls well inside the sample, not right at its noisy start. (`^IRX`, originally included for yield-curve slope, was dropped: Yahoo's endpoint returned it inconsistently for that one symbol — repeated identical requests alternated between the full history and a 19-point stub of just the last few weeks. Confirmed reproducible, not a one-off network blip, so not something worth building retry logic around for a secondary feature.)

**Features** (`features/engineering.py`): 16 causal columns — daily returns, 20-day realized vol, cross-ticker return spreads (breadth/small-cap/growth relative to `SPY`, credit stress as `HYG`-`LQD`), and 252-day rolling z-scores for level-type series (VIX, 10Y yield). Every feature at date *t* uses only data dated ≤ *t* — verified by a look-ahead-bias regression test in `tests/test_features.py` that perturbs future prices and checks past feature values don't move.

In [ ]:
panel, issues = load_many(list(config.TICKERS.keys()), tolerance_days=150, verbose=False)
print(f"loaded {panel['ticker'].nunique()} of {len(config.TICKERS)} tickers, {len(panel)} rows")
print(f"date range: {panel['date'].min().date()} -> {panel['date'].max().date()}")
if issues:
    print(f"\n{len(issues)} ticker(s) flagged (see README for why each is expected, not a bug):")
    for t, msgs in issues.items():
        for m in msgs:
            print(" -", m)

wide = panel.pivot(index="date", columns="ticker", values="adjclose")
wide = wide[list(config.TICKERS.keys())].ffill()

feat = build_raw_features(wide)
print(f"\nraw feature matrix: {feat.shape}, {feat.dropna().shape[0]} rows after warmup "
      f"(first clean row: {feat.dropna().index.min().date()})")
feat.dropna().describe().T[["mean", "std", "min", "max"]]

## 2 · Autoencoder: learning a regime embedding

Rather than hand-picking 3-4 "obviously important" features and feeding them straight to the HMM, a small encoder/decoder network (`models/autoencoder.py`) compresses the full 16-dimensional feature matrix into a **3-dimensional learned embedding**. This is the "representation learning" half of the pipeline, and the reason this project is more than "an HMM on VIX."

**Fit-on-train, freeze, apply-forward** — the same discipline used throughout the other two projects in this series, just applied here to a neural network instead of a walk-forward classifier: the feature scaler (mean/std) and the network's weights are both fit using only rows dated on or before `config.TRAIN_END` (2021-12-31). The frozen result is then applied to the full history, including 2022-2026, which the network never saw during training — a genuine holdout, not just a train/test shuffle. Verified with a look-ahead-bias regression test that perturbs holdout-period data and confirms the (deterministic, fixed-seed) training run produces bit-for-bit identical train-period latents either way.

A healthy result here is holdout reconstruction loss **comparable to, or only mildly worse than, train loss** — a big gap would mean the network memorized the training window rather than learning generalizable structure.

In [ ]:
ae = train_autoencoder(feat)

n_train = (feat.dropna().index <= ae["train_end"]).sum()
n_holdout = (feat.dropna().index > ae["train_end"]).sum()
print(f"train rows: {n_train}  |  holdout rows: {n_holdout}  (split at {ae['train_end'].date()})")
print(f"final train loss: {ae['train_loss_history'][-1]:.4f}")
print(f"holdout loss:      {ae['holdout_loss']:.4f}")
print("(comparable to train loss is healthy; a big gap would mean overfitting)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
axes[0].plot(ae["train_loss_history"], lw=1.2, color="#2a78d6")
axes[0].set_title("training loss (reconstruction MSE)")
axes[0].set_xlabel("epoch")
axes[0].axhline(ae["holdout_loss"], color="#eb6834", lw=1, ls="--", label="holdout loss")
axes[0].legend(frameon=False, fontsize=8)

latents = ae["latents"]
years = latents.index.year
sc = axes[1].scatter(latents["z0"], latents["z1"], c=years, cmap="viridis", s=4, alpha=0.5)
axes[1].set_title("latent space (z0 vs z1), colored by year")
axes[1].set_xlabel("z0"); axes[1].set_ylabel("z1")
plt.colorbar(sc, ax=axes[1], label="year")
plt.tight_layout()
plt.show()

## 3 · Fitting the regime HMM — and why BIC alone was misleading

A Gaussian HMM (`models/regime_hmm.py`) is fit on the frozen latent embedding — again using only the training window, then applied forward to the full history. The one open question is **how many states**.

The standard answer is "pick the state count that minimizes BIC." Here, that answer is actively wrong: BIC kept improving all the way through 7 states in the sweep below. Looking at *why* reveals the problem — past 4 states, the additional states have implied average durations of **1.5-2.5 days**, flickering back and forth between each other rather than persisting like genuine regimes. BIC was rewarding the model for fitting day-to-day noise as extra "regimes," not for finding real structure.

The fix: `select_n_states()` doesn't just minimize BIC — it picks the **largest state count where every single regime still has a plausible multi-day persistence** (≥5 trading days average). That's a duration-sanity constraint layered on top of the statistical fit criterion, and it's the same kind of cross-check used throughout this whole project series (never trust one metric in isolation).

In [ ]:
train_end = ae["train_end"]
sweep = sweep_n_states(latents, train_end, n_states_list=list(range(2, 8)))

sweep_table = pd.DataFrame([
    {"n_states": r["n_states"],
     "train_ll_per_obs": r["train_ll_per_obs"],
     "holdout_ll_per_obs": r["holdout_ll_per_obs"],
     "bic": r["bic"],
     "durations (days)": [round(d, 1) for d in implied_durations(r["model"])]}
    for r in sweep
])
print("BIC keeps improving through n_states=7 -- but watch the durations column:")
sweep_table

In [ ]:
best = select_n_states(sweep, min_duration_days=5.0)
model = best["model"]
print(f"selected n_states = {best['n_states']} "
      f"(largest state count where every regime still averages >=5 trading days)")

labels, probs = predict_regimes(model, latents)

REGIME_NAMES = {0: "Calm Bull", 1: "Crisis", 2: "Normal", 3: "Turbulent Bull"}
REGIME_COLORS = {0: "#1baf7a", 1: "#eb6834", 2: "#2a78d6", 3: "#eda100"}

print("\ntransition matrix (row = this regime, column = next):")
display(pd.DataFrame(model.transmat_.round(3),
                      index=[REGIME_NAMES[i] for i in range(model.n_components)],
                      columns=[REGIME_NAMES[i] for i in range(model.n_components)]))

print("\nimplied average duration per regime:")
for i, d in enumerate(implied_durations(model)):
    print(f"  {REGIME_NAMES[i]:16s} ~{d:.1f} trading days   ({(labels == i).sum()} days total)")

## 4 · Validation — no Sharpe ratio to lean on, so three different checks instead

With no P&L objective, "is this model good" needs answering a different way. Three checks, escalating from statistical to economic to robustness:

1. **Held-out log-likelihood** (above) — already checked as part of state selection: the chosen model's holdout likelihood is close to its train likelihood, not degraded.
2. **Historical alignment** — do the detected regimes land on real, known market events? The model was never told about any of these dates; if the rare, extreme-return state doesn't show up during 2008 and doesn't show up during anything else, that's a meaningful sanity check it passed by accident of the data, not by construction.
3. **Refit stability** — if the whole pipeline is retrained on progressively more history, do past regime labels stay put, or does the model keep rewriting history every time it sees a new year of data? A regime label that flips every time you refit isn't trustworthy no matter how good its likelihood looks.

In [ ]:
spy_ret = wide["SPY"].pct_change().reindex(labels.index)

print("regime characterization (SPY daily-return stats while in each regime):")
char = pd.DataFrame({
    "n_days": labels.value_counts().sort_index(),
    "spy_ann_ret": spy_ret.groupby(labels).mean() * 252,
    "spy_ann_vol": spy_ret.groupby(labels).std() * (252 ** 0.5),
})
char.index = [REGIME_NAMES[i] for i in char.index]
display(char)

print("\nregime composition during known crisis windows (model was never told these dates):")
for name, (start, end) in config.KNOWN_CRISIS_WINDOWS.items():
    window = labels.loc[start:end]
    if len(window) == 0:
        continue
    composition = window.value_counts(normalize=True).sort_index()
    comp_str = ", ".join(f"{REGIME_NAMES[k]}={v:.0%}" for k, v in composition.items())
    print(f"  {name:32s} ({start} to {end}): {comp_str}")

### Refit stability

The whole pipeline (autoencoder + HMM) is retrained from scratch at 4 progressively later `train_end` cutoffs — 2015, 2018, 2021, and today (2025-12-31, effectively the full sample) — and regime labels for a **shared 2009-2014 window** (inside every vintage's training data) are compared across vintages.

HMM state indices are arbitrary between independent fits — state 2 in one fit and state 0 in another can be the same regime — so states are matched across vintages by nearest centroid on their (mean SPY return, vol) characterization over the overlap window before comparing labels directly.

**This cell takes ~2 minutes** (4 full autoencoder + HMM-sweep refits).

In [ ]:
from scipy.optimize import linear_sum_assignment

VINTAGES = ["2015-12-31", "2018-12-31", "2021-12-31", "2025-12-31"]
OVERLAP_START, OVERLAP_END = "2009-01-01", "2014-12-31"

def characterize(lbls, ret):
    return pd.DataFrame({"mean_ret": ret.groupby(lbls).mean(), "vol": ret.groupby(lbls).std()})

def match_states(ref_char, other_char):
    ref = ref_char[["mean_ret", "vol"]].to_numpy()
    other = other_char[["mean_ret", "vol"]].to_numpy()
    cost = np.linalg.norm(ref[:, None, :] - other[None, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(cost)
    return dict(zip(other_char.index[col_ind], ref_char.index[row_ind]))

label_sets = {}
for vintage in VINTAGES:
    ae_v = train_autoencoder(feat, train_end=vintage)
    sweep_v = sweep_n_states(ae_v["latents"], pd.Timestamp(vintage), n_states_list=list(range(2, 7)))
    best_v = select_n_states(sweep_v, min_duration_days=5.0)
    lbls_v, _ = predict_regimes(best_v["model"], ae_v["latents"])
    label_sets[vintage] = lbls_v
    print(f"vintage {vintage}: selected n_states={best_v['n_states']}")

overlap_idx = None
for lbls in label_sets.values():
    idx = lbls.loc[OVERLAP_START:OVERLAP_END].index
    overlap_idx = idx if overlap_idx is None else overlap_idx.intersection(idx)

ref_vintage = VINTAGES[-1]
ref_labels = label_sets[ref_vintage].reindex(overlap_idx)
ref_char = characterize(ref_labels, spy_ret.reindex(overlap_idx))

print(f"\noverlap window: {overlap_idx.min().date()} -> {overlap_idx.max().date()}, {len(overlap_idx)} days\n")
for vintage in VINTAGES:
    lbls = label_sets[vintage].reindex(overlap_idx)
    char_v = characterize(lbls, spy_ret.reindex(overlap_idx))
    mapping = match_states(ref_char, char_v)
    matched = lbls.map(mapping)
    agree = (matched == ref_labels).mean()
    print(f"vintage train_end={vintage}: agreement with final vintage ({ref_vintage}) = {agree:.1%}")

## 5 · The regime timeline

SPY on a log scale, with the background shaded by detected regime. An interactive version of this same chart (with a hover crosshair and tooltip) is published as a standalone dashboard artifact — this is the static, notebook-native rendering.

In [ ]:
spy_px = wide["SPY"].reindex(labels.index)

fig, ax = plt.subplots(figsize=(13, 5))

# regime background bands, run-length encoded
dates = labels.index
lbl_vals = labels.to_numpy()
seg_start = 0
for i in range(1, len(lbl_vals) + 1):
    if i == len(lbl_vals) or lbl_vals[i] != lbl_vals[seg_start]:
        ax.axvspan(dates[seg_start], dates[i - 1], color=REGIME_COLORS[lbl_vals[seg_start]], alpha=0.18, lw=0)
        seg_start = i

ax.plot(spy_px.index, spy_px.values, color="#0b0b0b", lw=1.1)
ax.set_yscale("log")
ax.set_yticks([50, 100, 200, 400, 800])
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:.0f}"))
ax.set_title("SPY, shaded by detected regime")

legend_handles = [plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=10, label=REGIME_NAMES[k])
                   for k, c in REGIME_COLORS.items()]
ax.legend(handles=legend_handles, loc="upper left", frameon=False, ncol=4)
plt.tight_layout()
plt.show()

current_idx = probs.iloc[-1].to_numpy().argmax()
current_conf = probs.iloc[-1].to_numpy().max()
print(f"current regime (as of {probs.index[-1].date()}): "
      f"{REGIME_NAMES[current_idx]}  ({current_conf:.1%} model confidence)")

## 6 · Conclusion and limitations

**What this model does well:**
- Its rare, extreme-return "Crisis" state captures ~94% of the 2008 GFC and ~92% of the COVID crash — sharp, credit/liquidity-driven panics — while the milder Dec-2018 and 2015-16 selloffs land mostly in the calmer states, and the grinding, rate-hike-driven 2022 bear market lands almost entirely in a *different* elevated-vol-but-still-positive state. The model separates *panic crashes* from *grinding bear markets* without ever being told the difference — genuinely interesting unsupervised structure, not something engineered in.
- The duration-sanity model-selection rule (§3) caught BIC quietly overfitting noise as extra regimes — a concrete example of why one statistical criterion in isolation isn't enough.
- Refit-stability agreement (~82-84%) is moderate, not perfect: roughly 1 day in 6 gets relabeled as more data arrives, concentrated in transition/boundary periods rather than spread randomly — an expected property of unsupervised regime models, not a red flag, but worth disclosing precisely rather than rounding up to "stable."

**Limitations, stated plainly:**
- **No P&L objective, by design.** This says nothing about whether trading on these regime labels would be profitable — that's a different, harder question this project deliberately didn't ask.
- **A fixed 16-feature, 12-ticker universe.** Different macro inputs (credit default swaps, real yields, cross-currency bases) might carve regimes differently; this is one reasonable design, not the only one.
- **A single chronological train/holdout split**, not full walk-forward retraining like the other two projects — appropriate for validating a research model's statistical soundness, but a live deployment would need an actual retraining cadence.
- **State *names* (Calm Bull / Crisis / Normal / Turbulent Bull) are my own post-hoc economic labels** applied to what the model found, not something the HMM itself outputs — a purely statistical model has no notion of "bull" or "crisis," only clusters with distinct means and variances that happen to line up with those stories.

**Full interactive dashboard:** regime-colored chart with hover, transition heatmap, and crisis-alignment table — see `analysis/regime_dashboard.html` (also published as a standalone artifact).